**Recipe recommendation with two towers**

Only `new_dataset_recommendation_df`, loaded from `input_stage2/recipe_user_ratings.csv`,
is used. Run `map_recipe_nutrients.ipynb` first when rebuilding that CSV: each rating
must retain its original `RAW_interactions.csv` **date**.

The **item tower** encodes separate ingredient, adjective, and verb semantic vectors,
per-recipe nutrition means, nutrition-cluster membership and cluster means, and summed
ingredient prices with missing-price uncertainty. The **user tower** encodes the user's
past recipe ratings: recent history has more weight than old history, and high and low
ratings contribute to separate preference summaries. Both towers learn a common vector
space; their dot product produces a calibrated predicted rating from 1 to 5.

```text
Ingredient / adjective / verb semantics + nutrition clusters + price → item MLP → item vector
Past recipe features + ratings + exponential date decay             → user MLP → user vector
                                                                                │
                           predicted rating ← sigmoid calibration ← dot product ┘
```

This is supervised explicit-rating learning. Unrated recipes remain candidates but
never become zero-rating training labels. No user-ID embedding is required.

In [ ]:
# Run once if these packages are missing, then restart the notebook kernel.
# %pip install torch transformers==5.17.0 numpy pandas scikit-learn joblib matplotlib
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display

**Configuration**

`RECENCY_HALF_LIFE_DAYS = 180` gives a rating half its history weight after 180 days.
Each missing price contributes **±$2**, so three null ingredients contribute ±$6 around
the sum of known prices (lower bound clipped at zero). This is an uncertainty feature,
not an imputed ingredient cost. The item tower receives both the known sum and the
uncertainty. Nested fuzzy-price dictionaries are summed recursively.

Nutrition columns retain the input CSV's existing names, units, and values. Each column
is averaged separately per recipe; a list-valued cell is reduced to its nonmissing mean.
Duplicate interaction rows do not multiply a recipe's weight in clustering. Median
imputation, missing indicators, `log1p`, and scaling precede nutrition-only clustering.
Every recipe has a cluster; candidates are not restricted to the user's prior clusters.

In [ ]:
RANDOM_STATE = 42
BERT_MODEL = "bert-base-uncased"
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
BERT_BATCH_SIZE = 64
BERT_MAX_LENGTH = 64
SEMANTIC_DIM_PER_FIELD = 128
NUTRITION_CLUSTERS = 32
MISSING_PRICE_ERROR_DOLLARS = 2.0
RECENCY_HALF_LIFE_DAYS = 180.0
MAX_HISTORY = 64
VALIDATION_FRACTION = 0.2
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 4

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
NOTEBOOK_DIR = next((p for p in (Path.cwd(), Path.cwd() / "DataMappingRecommendation")
                     if (p / "input_stage2/recipe_user_ratings.csv").is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Run from DataMappingRecommendation or its project root; restore recipe_user_ratings.csv.")
DATA_PATH = NOTEBOOK_DIR / "input_stage2/recipe_user_ratings.csv"
ARTIFACT_DIR = NOTEBOOK_DIR / "model_artifacts/two_tower"
CACHE_DIR = NOTEBOOK_DIR / "output/two_tower_cache"
print(f"Device: {device}; recipe data: {DATA_PATH}")

**Self-contained model and data definitions**

The following two cells contain parsing, feature construction, both towers, chronological
history assembly, training, and prediction. No local model module is required.

In [ ]:
"""Two-tower rating model with strictly historical, recency-weighted profiles."""
import copy
import math
import warnings
from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


def canonical_id(value):
    """Normalize integer IDs loaded as numbers, strings, or CSV float strings."""
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    if value.lower() in {"", "nan", "none", "null", "<na>"}:
        return None
    try:
        number = Decimal(value)
        if not number.is_finite():
            return None
        if number == number.to_integral_value():
            return str(int(number))
    except InvalidOperation:
        pass
    return value


def build_interactions(df, catalog):
    required = {"recipe_id", "user_id", "rating", "date"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing interaction columns {sorted(missing)}. Rerun map_recipe_nutrients.ipynb to export the original rating date.")
    catalog_ids = catalog["recipe_id"].map(canonical_id)
    if catalog_ids.isna().any() or catalog_ids.duplicated().any():
        raise ValueError("Catalog recipe_id values must be valid and unique after ID normalization.")
    frame = df.loc[:, ["recipe_id", "user_id", "rating", "date"]].copy()
    ratings = pd.to_numeric(frame["rating"], errors="coerce")
    malformed = frame["rating"].notna() & ratings.isna()
    invalid = ratings.notna() & ratings.ne(0) & ~ratings.between(1, 5)
    if malformed.any() or invalid.any():
        raise ValueError("Ratings must be numeric values from 1 to 5; use 0 or null for unlabeled rows.")
    frame["rating"] = ratings
    frame = frame.loc[ratings.between(1, 5)].copy()
    frame["user_id"] = frame["user_id"].map(canonical_id)
    frame["recipe_id"] = frame["recipe_id"].map(canonical_id)
    valid_ids = frame["user_id"].notna() & frame["recipe_id"].isin(set(catalog_ids))
    if (~valid_ids).any():
        warnings.warn(f"Excluded {(~valid_ids).sum():,} labeled rows with missing user IDs or recipes absent from the catalog.", stacklevel=2)
    frame = frame.loc[valid_ids].copy()
    frame["date"] = pd.to_datetime(frame["date"], format="mixed", errors="coerce", utc=True)
    if frame["date"].isna().any():
        raise ValueError("Labeled interactions contain missing/invalid dates. Rerun map_recipe_nutrients.ipynb and regenerate the mapped CSV with original dates; do not invent timestamps.")
    item_lookup = dict(zip(catalog_ids, range(len(catalog_ids))))
    frame["item_index"] = frame["recipe_id"].map(item_lookup).astype("int64")
    frame = frame.sort_values("date", kind="stable").drop_duplicates(["user_id", "recipe_id"], keep="last")
    return frame[["user_id", "item_index", "rating", "date"]].reset_index(drop=True)


def temporal_split(interactions, validation_fraction=0.2):
    """Hold out newest UTC calendar-day blocks per user; retain a training day."""
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must lie between 0 and 1.")
    frame = interactions.sort_values(["user_id", "date"], kind="stable").reset_index(drop=True)
    days = frame["date"].dt.floor("D")
    validation_mask = np.zeros(len(frame), dtype=bool)
    for _, group in frame.groupby("user_id", sort=False):
        day_counts = days.loc[group.index].value_counts().sort_index()
        if len(day_counts) < 2:
            continue
        desired = max(1, math.ceil(len(group) * validation_fraction))
        newest_counts = day_counts.iloc[1:][::-1].cumsum()
        boundary_position = min(int(np.searchsorted(newest_counts.to_numpy(), desired)), len(newest_counts) - 1)
        boundary = newest_counts.index[boundary_position]
        validation_mask[group.index] = days.loc[group.index].ge(boundary)
    return frame.loc[~validation_mask].reset_index(drop=True), frame.loc[validation_mask].reset_index(drop=True)


class HistoryIndex:
    def __init__(self, history):
        self.users = {}
        for user, group in history.sort_values("date", kind="stable").groupby("user_id", sort=False):
            self.users[canonical_id(user)] = (
                group["date"].astype("int64").to_numpy(),
                group["item_index"].to_numpy(dtype=np.int64),
                group["rating"].to_numpy(dtype=np.float32),
            )


def _utc_day(value):
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        raise ValueError("A valid as_of/reference_date is required.")
    return (stamp.tz_localize("UTC") if stamp.tzinfo is None else stamp.tz_convert("UTC")).floor("D")


def user_profile(history, item_features, user_id, as_of, max_history=64, half_life_days=180, exclude_item=None):
    """Pool only earlier UTC days; never condition a profile on candidate features."""
    if max_history < 1 or half_life_days <= 0:
        raise ValueError("max_history and half_life_days must be positive.")
    index = history if isinstance(history, HistoryIndex) else HistoryIndex(history)
    features = np.asarray(item_features, dtype=np.float32)
    profile = np.zeros(2 * features.shape[1] + 3, dtype=np.float32)
    records = index.users.get(canonical_id(user_id))
    cutoff = _utc_day(as_of).value
    if records is None:
        return profile
    dates, items, ratings = records
    end = np.searchsorted(dates, cutoff, side="left")
    start = max(0, end - max_history)
    selected = np.arange(start, end, dtype=np.int64)
    if exclude_item is not None:
        selected = selected[items[selected] != exclude_item]
        while len(selected) < max_history and start > 0:
            previous = max(0, start - (max_history - len(selected)))
            extra = np.arange(previous, start, dtype=np.int64)
            selected = np.concatenate([extra[items[extra] != exclude_item], selected])
            start = previous
    selected = selected[-max_history:]
    if not len(selected):
        return profile
    age_days = (cutoff - dates[selected]) / (86400 * 1e9)
    # Relative ages preserve normalized weights and avoid underflow for old data.
    decay = np.exp2(-(age_days - age_days.min()) / half_life_days)
    normalized = (ratings[selected] - 1.0) / 4.0
    dim = features.shape[1]
    for offset, strength in [(0, normalized), (dim, 1.0 - normalized)]:
        weights = decay * strength
        if weights.sum() > 0:
            profile[offset:offset + dim] = np.average(features[items[selected]], axis=0, weights=weights)
    profile[-3:] = [np.average(normalized, weights=decay), np.log1p(len(selected)), 1.0]
    return profile


class HistoryDataset(Dataset):
    def __init__(self, targets, history, item_features, max_history=64, half_life_days=180,
                 training=True, reference_date=None, weight_floor=0.1):
        self.targets = targets.reset_index(drop=True).copy()
        self.history_index = HistoryIndex(history)
        self.item_features = np.asarray(item_features, dtype=np.float32)
        self.max_history, self.half_life_days = max_history, half_life_days
        if self.item_features.ndim != 2 or not np.isfinite(self.item_features).all():
            raise ValueError("item_features must be a finite, two-dimensional feature matrix.")
        if max_history < 1 or half_life_days <= 0 or not 0 < weight_floor <= 1:
            raise ValueError("Use max_history >= 1, half_life_days > 0, and 0 < weight_floor <= 1.")
        self.user_feature_dim = 2 * self.item_features.shape[1] + 3
        self.weights = np.ones(len(self.targets), dtype=np.float32)
        if training and len(self.targets):
            reference = history["date"].max() if reference_date is None else reference_date
            reference = _utc_day(reference)
            age = (reference - self.targets["date"].dt.floor("D")).dt.total_seconds().to_numpy() / 86400
            self.weights = np.maximum(weight_floor, np.exp2(-np.maximum(age, 0) / half_life_days)).astype(np.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        row = self.targets.iloc[index]
        profile = user_profile(self.history_index, self.item_features, row.user_id, row.date,
                               self.max_history, self.half_life_days, exclude_item=row.item_index)
        return {"user_features": torch.from_numpy(profile), "item_index": int(row.item_index),
                "rating": torch.tensor(row.rating, dtype=torch.float32),
                "sample_weight": torch.tensor(self.weights[index], dtype=torch.float32)}


class TwoTowerModel(nn.Module):
    def __init__(self, feature_dim, user_feature_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()
        def tower(input_dim):
            return nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
                                 nn.ReLU(), nn.Linear(hidden_dim, embedding_dim))
        self.item_tower = tower(feature_dim)
        self.user_tower = tower(user_feature_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(4.0)))
        self.logit_bias = nn.Parameter(torch.tensor(0.0))

    def encode_items(self, features):
        return F.normalize(self.item_tower(features), p=2, dim=-1)

    def encode_users(self, features):
        return F.normalize(self.user_tower(features), p=2, dim=-1)

    def score_embeddings(self, users, items):
        similarity = (users * items).sum(dim=-1)
        return 1.0 + 4.0 * torch.sigmoid(self.logit_scale.exp().clamp(max=100) * similarity + self.logit_bias)

    def forward(self, user_features, item_features):
        return self.score_embeddings(self.encode_users(user_features), self.encode_items(item_features))


def _rating_metrics(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    return {"rmse": float(np.sqrt(np.mean((actual - predicted) ** 2))),
            "mae": float(np.mean(np.abs(actual - predicted)))} if len(actual) else {"rmse": float("nan"), "mae": float("nan")}


def evaluate_baselines(train, validation):
    """All baseline estimates are fit on training ratings only."""
    global_mean = float(train["rating"].mean())
    predictions = {"global_mean": np.full(len(validation), global_mean)}
    for column, name in [("user_id", "user_mean"), ("item_index", "item_mean")]:
        means = train.groupby(column)["rating"].mean()
        predictions[name] = validation[column].map(means).fillna(global_mean).to_numpy()
    return pd.DataFrame([{"model": name, **_rating_metrics(validation["rating"], values)}
                         for name, values in predictions.items()])


@torch.no_grad()
def evaluate_two_tower(model, dataset, item_features, batch_size=512, device=None):
    device = torch.device(device) if device is not None else next(model.parameters()).device
    features = torch.as_tensor(np.asarray(item_features, dtype=np.float32))
    actual, predicted = [], []
    model.eval()
    for batch in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        scores = model(batch["user_features"].to(device), features[batch["item_index"]].to(device))
        actual.extend(batch["rating"].numpy())
        predicted.extend(scores.cpu().numpy())
    return _rating_metrics(actual, predicted)


def train_two_tower(model, train_dataset, validation_dataset, item_features, epochs=15,
                    batch_size=256, learning_rate=1e-3, device=None, patience=4):
    if not len(train_dataset) or epochs < 1:
        raise ValueError("Training requires at least one labeled interaction and one epoch.")
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model.to(device)
    features = torch.as_tensor(np.asarray(item_features, dtype=np.float32))
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    # A fixed normalizer preserves recency weighting even for a one-row batch.
    mean_train_weight = float(np.mean(train_dataset.weights))
    best_score, best_state, stale, records = float("inf"), None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        squared_error, total_weight = 0.0, 0.0
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            prediction = model(batch["user_features"].to(device), features[batch["item_index"]].to(device))
            weights = batch["sample_weight"].to(device)
            errors = (prediction - batch["rating"].to(device)).square()
            loss = (weights * errors).mean() / mean_train_weight
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            squared_error += float((weights * errors).sum().detach().cpu())
            total_weight += float(weights.sum().cpu())
        validation = evaluate_two_tower(model, validation_dataset, features, batch_size, device)
        train_rmse = math.sqrt(squared_error / total_weight)
        records.append({"epoch": epoch, "train_weighted_rmse": train_rmse,
                        "validation_rmse": validation["rmse"], "validation_mae": validation["mae"]})
        print(f"Epoch {epoch:02d}: weighted train RMSE={train_rmse:.4f}; validation RMSE={validation['rmse']:.4f}")
        score = validation["rmse"] if len(validation_dataset) else train_rmse
        if not math.isfinite(score):
            raise RuntimeError("Training produced non-finite loss; check item features and labels.")
        if score < best_score:
            best_score, best_state, stale = score, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    metrics = evaluate_two_tower(model, validation_dataset, features, batch_size, device)
    return model, pd.DataFrame(records), metrics


@torch.no_grad()
def encode_catalog(model, item_features, batch_size=1024, device=None):
    device = torch.device(device) if device is not None else next(model.parameters()).device
    model.eval()
    features = torch.as_tensor(np.asarray(item_features, dtype=np.float32))
    return torch.cat([model.encode_items(batch.to(device)).cpu()
                      for batch in features.split(batch_size)], dim=0)


@torch.no_grad()
def recommend_recipes(model, catalog, item_features, interactions, user_id, top_k=10,
                      as_of=None, max_history=64, half_life_days=180, item_embeddings=None):
    if top_k < 1:
        raise ValueError("top_k must be positive.")
    # An explicit cutoff prevents future ratings from affecting historical recommendations.
    cutoff = _utc_day(as_of) if as_of is not None else _utc_day(pd.Timestamp.now(tz="UTC"))
    user_key = canonical_id(user_id)
    observed = interactions.loc[
        (interactions["date"] < cutoff)
        & interactions["user_id"].map(canonical_id).eq(user_key)
    ]
    profile = user_profile(observed, item_features, user_id, cutoff, max_history, half_life_days)
    device = next(model.parameters()).device
    model.eval()
    user_embedding = model.encode_users(torch.from_numpy(profile).unsqueeze(0).to(device)).cpu()
    embeddings = encode_catalog(model, item_features) if item_embeddings is None else torch.as_tensor(item_embeddings).cpu()
    similarity = (embeddings * user_embedding).sum(dim=-1)
    scores = 1.0 + 4.0 * torch.sigmoid(model.logit_scale.detach().cpu().exp().clamp(max=100) * similarity + model.logit_bias.detach().cpu())
    seen = observed["item_index"].to_numpy(dtype=np.int64)
    scores[seen] = -torch.inf
    eligible = torch.isfinite(scores).sum().item()
    selected = torch.topk(scores, min(top_k, eligible)).indices.numpy()
    columns = [column for column in catalog.columns if column in {"recipe_id", "name"} or "price" in column or "cost" in column]
    result = catalog.iloc[selected][columns].copy()
    result["predicted_rating"] = scores[selected].numpy()
    return result.reset_index(drop=True)

In [ ]:
# Content features are computed once per recipe; rating frequency never weights them.
import ast
import hashlib
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from transformers import AutoModel, AutoTokenizer

NUTRITION_COLUMNS = [
    'calories (g)', 'total fat (g)', 'sugar (g)', 'sodium (g)',
    'protein (g)', 'saturated fat (g)', 'carbohydrates (g)',
]
TEXT_FIELDS = ('ingredient_terms', 'adj_terms', 'verb_terms')
PRICE_FEATURE_COLUMNS = ['price_total_known', 'price_error_dollars',
                         'price_missing_fraction', 'price_all_missing']


@lru_cache(maxsize=100_000)
def parse_literal(text):
    text = text.strip()
    if text.lower() in {'', 'none', 'null', 'nan', '<na>'}:
        return None
    try:
        return json.loads(text)
    except (ValueError, TypeError):
        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            if text.startswith(('[', '{', '(')):
                raise ValueError(f'Malformed serialized feature: {text[:100]!r}')
            return text


def flatten_terms(value):
    if isinstance(value, str):
        parsed = parse_literal(value)
        if isinstance(parsed, str):
            return [parsed.strip().lower()] if parsed.strip() else []
        return flatten_terms(parsed)
    if isinstance(value, (list, tuple, np.ndarray)):
        return [term for part in value for term in flatten_terms(part)]
    if value is None or value is pd.NA or (np.isscalar(value) and pd.isna(value)):
        return []
    raise ValueError(f'Expected ingredient/adjective/verb text, got {value!r}')


def nutrition_mean(value):
    def numbers(part):
        if isinstance(part, str):
            part = parse_literal(part)
        if isinstance(part, (list, tuple, np.ndarray)):
            return [number for child in part for number in numbers(child)]
        if part is None or pd.isna(part):
            return []
        number = float(part)
        if not np.isfinite(number):
            return []
        if number < 0:
            raise ValueError('Nutrition values must be nonnegative.')
        return [number]
    values = numbers(value)
    return float(np.mean(values)) if values else np.nan


def summarize_price(value, ingredient_count, missing_price_error=2.0):
    """Each null leaf contributes +/- $2 uncertainty, never a fabricated price."""
    if not np.isfinite(missing_price_error) or missing_price_error < 0:
        raise ValueError('missing_price_error must be finite and nonnegative.')
    payload = parse_literal(value) if isinstance(value, str) else value
    if not isinstance(payload, (dict, list, tuple)) and pd.isna(payload):
        payload = None
    def leaves(node):
        if isinstance(node, dict):
            return [leaf for child in node.values() for leaf in leaves(child)] or [None]
        if isinstance(node, (list, tuple)):
            return [leaf for child in node for leaf in leaves(child)] or [None]
        if node is None or pd.isna(node):
            return [None]
        number = float(node)
        if not np.isfinite(number):
            return [None]
        if number < 0:
            raise ValueError('Ingredient prices must be nonnegative.')
        return [number]
    if payload is None or (isinstance(payload, (dict, list)) and not payload):
        values = [None] * max(1, ingredient_count)
    else:
        if not isinstance(payload, (dict, list)):
            raise ValueError('price_breakdown must be a JSON dictionary/list or null.')
        values = leaves(payload)
    known = [price for price in values if price is not None]
    missing = len(values) - len(known)
    total = float(sum(known))
    error = float(missing_price_error * missing)
    return {'price_total_known': total, 'price_missing_count': missing,
            'price_error_dollars': error, 'price_lower': max(0., total - error),
            'price_upper': total + error, 'price_missing_fraction': missing / len(values),
            'price_all_missing': int(not known)}


def build_recipe_catalog(frame, missing_price_error=2.0):
    required = {'recipe_id', 'ingredients', 'adj', 'verb', 'price_breakdown', *NUTRITION_COLUMNS}
    if missing := required - set(frame):
        raise ValueError(f'Recipe CSV is missing columns: {sorted(missing)}. Rerun the mapping notebook.')
    # Work only on content columns, avoiding a second copy of all interaction rows.
    text_column = next((c for c in ('product', 'products') if c in frame), 'ingredients')
    columns = list(dict.fromkeys(['recipe_id', 'name', 'ingredients', text_column,
                                  'adj', 'verb', 'price_breakdown', *NUTRITION_COLUMNS]))
    columns = [column for column in columns if column in frame]
    recipes = frame[columns].drop_duplicates().copy()
    recipes['recipe_id'] = recipes['recipe_id'].map(canonical_id)
    if recipes['recipe_id'].isna().any():
        raise ValueError('Recipe IDs cannot be missing.')
    # Text/prices should be invariant across repeated rating rows for a recipe.
    content_columns = list(dict.fromkeys(['ingredients', text_column, 'adj', 'verb', 'price_breakdown']))
    conflicts = recipes.groupby('recipe_id')[content_columns].nunique(dropna=False).gt(1).any(axis=1)
    if conflicts.any():
        raise ValueError(f'Conflicting content for recipe IDs: {conflicts[conflicts].index[:5].tolist()}')
    for column in NUTRITION_COLUMNS:
        recipes[column] = recipes[column].map(nutrition_mean)
    # Average each nutrient independently per recipe, never mix unlike nutrient units.
    nutrient_means = recipes.groupby('recipe_id', sort=False)[NUTRITION_COLUMNS].mean()
    catalog = recipes.drop_duplicates('recipe_id').set_index('recipe_id')
    catalog[NUTRITION_COLUMNS] = nutrient_means.reindex(catalog.index)
    catalog = catalog.reset_index()
    if 'name' not in catalog:
        catalog['name'] = catalog['recipe_id']
    catalog['ingredient_terms'] = [
        flatten_terms(entities) or flatten_terms(original)
        for entities, original in zip(catalog[text_column], catalog['ingredients'])]
    for column in ('adj', 'verb'):
        catalog[f'{column}_terms'] = catalog[column].map(flatten_terms)
    price_rows = [summarize_price(value, len(set(terms)), missing_price_error)
                  for value, terms in zip(catalog['price_breakdown'], catalog['ingredient_terms'])]
    return pd.concat([catalog, pd.DataFrame(price_rows)], axis=1)


def encode_semantic_fields(catalog, cache_dir, device, model_name='bert-base-uncased',
                           revision='86b5e0934494bd15c9632b12f734a8a67f723594',
                           batch_size=64, max_length=64, semantic_dim=128, seed=42):
    """Frozen phrase BERT -> shared SVD -> separate mean-pooled text fields."""
    if len(catalog) == 0:
        raise ValueError('No recipes to encode.')
    terms = sorted({term for column in TEXT_FIELDS for row in catalog[column] for term in row})
    if not terms:
        raise ValueError('No ingredient/adjective/verb text found in the recipe catalog.')
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    identity = {'model': model_name, 'revision': revision, 'max_length': max_length,
                'pooling': 'mean_without_padding_or_special_tokens', 'terms': terms}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    cache_path = cache_dir / f'bert_phrases_{key}.npy'
    if cache_path.is_file():
        vectors = np.load(cache_path, allow_pickle=False)
        if vectors.ndim != 2 or len(vectors) != len(terms) or not np.isfinite(vectors).all():
            raise ValueError(f'Invalid phrase cache: {cache_path}')
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision)
        encoder = AutoModel.from_pretrained(model_name, revision=revision).to(device).eval()
        encoder.requires_grad_(False)
        vectors = np.empty((len(terms), encoder.config.hidden_size), dtype=np.float32)
        with torch.inference_mode():
            for start in range(0, len(terms), batch_size):
                encoded = tokenizer(terms[start:start + batch_size], padding=True, truncation=True,
                                    max_length=max_length, return_special_tokens_mask=True, return_tensors='pt')
                special = encoded.pop('special_tokens_mask').to(device)
                encoded = {key: value.to(device) for key, value in encoded.items()}
                hidden = encoder(**encoded).last_hidden_state
                mask = (encoded['attention_mask'].bool() & ~special.bool()).unsqueeze(-1)
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
                vectors[start:start + batch_size] = torch.nn.functional.normalize(pooled, dim=-1).cpu().numpy()
                if start % (batch_size * 50) == 0:
                    print(f'BERT phrases: {min(start + batch_size, len(terms)):,}/{len(terms):,}')
        np.save(cache_path, vectors, allow_pickle=False)
        del encoder
    # Compact shared phrase space bounds full-catalog and history memory use.
    components = min(semantic_dim, vectors.shape[1], max(1, len(vectors) - 1))
    reducer = None
    if len(vectors) > 1 and components < vectors.shape[1]:
        reducer = TruncatedSVD(n_components=components, random_state=seed)
        phrase_vectors = normalize(reducer.fit_transform(vectors)).astype(np.float32)
    else:
        phrase_vectors = vectors.astype(np.float32, copy=False)
    lookup = {term: i for i, term in enumerate(terms)}
    fields = []
    present = np.zeros((len(catalog), len(TEXT_FIELDS)), dtype=np.float32)
    for field_id, column in enumerate(TEXT_FIELDS):
        pooled = np.zeros((len(catalog), phrase_vectors.shape[1]), dtype=np.float32)
        for row_id, row_terms in enumerate(catalog[column]):
            if row_terms:
                pooled[row_id] = phrase_vectors[[lookup[term] for term in row_terms]].mean(axis=0)
                present[row_id, field_id] = 1.
        fields.append(normalize(pooled))
    return np.concatenate([*fields, present], axis=1).astype(np.float32), {
        'model_name': model_name, 'revision': revision, 'max_length': max_length,
        'semantic_dim': phrase_vectors.shape[1], 'svd': reducer,
        'text_fields': TEXT_FIELDS, 'phrase_cache_key': key,
    }


def build_numeric_features(catalog, n_clusters=32, seed=42):
    raw = catalog[NUTRITION_COLUMNS].to_numpy(dtype=np.float64)
    missing = ~np.isfinite(raw)
    medians = catalog[NUTRITION_COLUMNS].median().fillna(0.).to_numpy(dtype=np.float64)
    filled = np.where(missing, medians, raw)
    nutrition_scaler = StandardScaler()
    scaled = nutrition_scaler.fit_transform(np.log1p(filled)).astype(np.float32)
    distinct = len(np.unique(scaled, axis=0))
    count = max(1, min(n_clusters, len(catalog), distinct))
    clusterer = MiniBatchKMeans(n_clusters=count, random_state=seed, n_init=10,
                               batch_size=max(256, count * 3))
    labels = clusterer.fit_predict(scaled)
    catalog['nutrition_cluster'] = labels
    one_hot = np.eye(count, dtype=np.float32)[labels]
    cluster_means = pd.DataFrame(filled, columns=NUTRITION_COLUMNS).groupby(labels).mean()
    cluster_means.index.name = 'nutrition_cluster'
    # Actual per-column means over one row per recipe, not k-means' approximate centers.
    mean_features = nutrition_scaler.transform(np.log1p(cluster_means.loc[labels].to_numpy())).astype(np.float32)
    prices = catalog[PRICE_FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    prices[:, :2] = np.log1p(prices[:, :2])
    price_scaler = StandardScaler()
    price_features = price_scaler.fit_transform(prices).astype(np.float32)
    numeric = np.concatenate([scaled, missing.astype(np.float32), one_hot, mean_features, price_features], axis=1)
    if not np.isfinite(numeric).all():
        raise ValueError('Item features contain non-finite values.')
    return numeric, cluster_means, {'nutrition_columns': NUTRITION_COLUMNS,
        'nutrition_medians': medians, 'nutrition_scaler': nutrition_scaler,
        'nutrition_clusterer': clusterer, 'cluster_means': cluster_means,
        'price_scaler': price_scaler, 'price_feature_columns': PRICE_FEATURE_COLUMNS}

In [ ]:
new_dataset_recommendation_df = pd.read_csv(
    DATA_PATH, dtype={"recipe_id": "string", "user_id": "string"}, low_memory=False
)
if "date" not in new_dataset_recommendation_df:
    raise ValueError("The CSV has no date column. Rerun map_recipe_nutrients.ipynb to export original interaction dates.")
item_catalog = build_recipe_catalog(
    new_dataset_recommendation_df, missing_price_error=MISSING_PRICE_ERROR_DOLLARS
)
observed_ratings = build_interactions(new_dataset_recommendation_df, item_catalog)
train_ratings, validation_ratings = temporal_split(
    observed_ratings, validation_fraction=VALIDATION_FRACTION
)
print(f"{len(new_dataset_recommendation_df):,} source rows; {len(item_catalog):,} distinct recipes")
print(f"{len(train_ratings):,} training ratings; {len(validation_ratings):,} validation ratings")
display(new_dataset_recommendation_df.head())
display(observed_ratings["rating"].value_counts().sort_index().rename("observed_count").to_frame())

**Frozen phrase semantics and nutrition clusters**

Each distinct phrase is encoded once by the pinned BERT checkpoint. Padding and special
tokens are excluded from mean pooling. A shared SVD compresses the frozen phrase vectors;
then each recipe separately mean-pools its ingredient, adjective, and verb phrases.
These three vectors and field-presence indicators remain separate item inputs.

Content preprocessing uses the complete recipe catalog, including unrated recipes, but
never reads user IDs, ratings, or dates. Validation therefore measures future ratings for
a known catalog, not a future-catalog cold-start scenario. Nutrition clusters use scaled
per-recipe nutrient means, and the displayed cluster means count each recipe once.

References: [BERT](https://huggingface.co/docs/transformers/model_doc/bert) and
[MiniBatchKMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MiniBatchKMeans.html).

In [ ]:
semantic_features, semantic_preprocessing = encode_semantic_fields(
    item_catalog, CACHE_DIR, device, model_name=BERT_MODEL, revision=BERT_REVISION,
    batch_size=BERT_BATCH_SIZE, max_length=BERT_MAX_LENGTH,
    semantic_dim=SEMANTIC_DIM_PER_FIELD, seed=RANDOM_STATE
)
numeric_features, nutrition_cluster_means, numeric_preprocessing = build_numeric_features(
    item_catalog, n_clusters=NUTRITION_CLUSTERS, seed=RANDOM_STATE
)
item_features = np.concatenate([semantic_features, numeric_features], axis=1).astype(np.float32)
del semantic_features, numeric_features
assert np.isfinite(item_features).all()
print(f"Item feature matrix: {item_features.shape}; {item_features.nbytes / 2**20:.1f} MiB")
display(nutrition_cluster_means)
display(item_catalog[["recipe_id", "name", "nutrition_cluster", "price_total_known",
                      "price_missing_count", "price_error_dollars", "price_lower", "price_upper"]].head())

**Chronological supervision with recent ratings weighted more heavily**

Ratings 1–5 are observed labels; 0 and missing ratings are unrated. Missing or invalid
dates on observed ratings raise an error. Repeated user–recipe ratings keep their latest
event. Validation holds out each eligible user's most recent date blocks, with at least
one earlier block retained for training. Users with only one date block have no holdout.

For every target, the user tower sees **only strictly earlier training ratings**, with the
target recipe excluded. Same-day events have no known order, so none can reveal another
same-day target. Validation ratings never enter training histories or validation histories.

A maximum of 64 recent observations contributes to two pools, weighted by both date decay
and rating strength: `(rating - 1) / 4` for the high-rating pool, `(5 - rating) / 4` for the
low-rating pool. The profile also contains weighted mean rating, history size, and a
history-presence flag. Empty history produces a learned general preference profile.
Training loss also gives more weight to recent targets, relative to the **training-only**
latest date; a 0.1 floor retains signal from older observations. Validation RMSE/MAE are
unweighted on the original 1–5 scale. This per-user split is not a single global time cutoff.

In [ ]:
train_dataset = HistoryDataset(
    train_ratings, train_ratings, item_features, max_history=MAX_HISTORY,
    half_life_days=RECENCY_HALF_LIFE_DAYS, training=True
)
validation_dataset = HistoryDataset(
    validation_ratings, train_ratings, item_features, max_history=MAX_HISTORY,
    half_life_days=RECENCY_HALF_LIFE_DAYS, training=False
)
recommendation_model = TwoTowerModel(
    feature_dim=item_features.shape[1], user_feature_dim=2 * item_features.shape[1] + 3,
    embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM
)
print(recommendation_model)
recommendation_model, training_history, validation_metrics = train_two_tower(
    recommendation_model, train_dataset, validation_dataset, item_features,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    device=device, patience=EARLY_STOPPING_PATIENCE
)
display(training_history)
display(pd.Series(validation_metrics, name="value").to_frame())
baseline_metrics = evaluate_baselines(train_ratings, validation_ratings)
display(baseline_metrics)

**Recommend unseen recipes**

After validation is measured, serving may use all genuinely observed ratings. The example
uses a cutoff after the dataset's last interaction, appropriate for this historical data.
Set `as_of` explicitly to score from an earlier history; same-day/future ratings are excluded.
Candidates with no ratings are still supported by item content. Predicted ratings are model
scores, not new observed labels. RMSE/MAE evaluate rating accuracy, not top-k retrieval quality.

In [ ]:
example_user_id = observed_ratings["user_id"].value_counts().index[0]
recommendations = recommend_recipes(
    recommendation_model, item_catalog, item_features, observed_ratings,
    user_id=example_user_id, top_k=20, max_history=MAX_HISTORY,
    half_life_days=RECENCY_HALF_LIFE_DAYS,
    as_of=observed_ratings["date"].max().floor("D") + pd.Timedelta(days=1)
)
print(f"Recipe recommendations for user {example_user_id}")
display(recommendations)

**Save the model together with its serving features**

The artifact directory contains both towers, the frozen preprocessing objects, ordered
recipe catalog and features, precomputed item vectors, observed histories, and configuration.
Keep them together: item index order must match across the catalog, arrays, and histories.
The reload example below uses the saved catalog directly and does not rerun BERT or clustering.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_config = {"feature_dim": item_features.shape[1],
                "user_feature_dim": 2 * item_features.shape[1] + 3,
                "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM}
torch.save({"state_dict": {key: value.detach().cpu() for key, value in recommendation_model.state_dict().items()},
            "model_config": model_config}, ARTIFACT_DIR / "two_tower.pt")
np.save(ARTIFACT_DIR / "item_features.npy", item_features, allow_pickle=False)
item_vectors = encode_catalog(recommendation_model, item_features, device=device)
np.save(ARTIFACT_DIR / "item_vectors.npy", item_vectors, allow_pickle=False)
serving_columns = ["recipe_id", "name", "nutrition_cluster", *NUTRITION_COLUMNS,
                   "price_total_known", "price_missing_count", "price_error_dollars", "price_lower", "price_upper"]
item_catalog[serving_columns].to_csv(ARTIFACT_DIR / "catalog.csv", index=False)
observed_ratings.to_csv(ARTIFACT_DIR / "observed_ratings.csv", index=False)
joblib.dump({"semantic": semantic_preprocessing, "numeric": numeric_preprocessing},
            ARTIFACT_DIR / "content_preprocessing.joblib")
config = {"format_version": "recipe_two_tower_v1", "model_config": model_config,
          "bert_model": BERT_MODEL, "bert_revision": BERT_REVISION,
          "recency_half_life_days": RECENCY_HALF_LIFE_DAYS, "max_history": MAX_HISTORY,
          "missing_price_error_dollars": MISSING_PRICE_ERROR_DOLLARS,
          "random_state": RANDOM_STATE, "validation_fraction": VALIDATION_FRACTION,
          "source_csv": str(DATA_PATH), "rating_min": 1.0, "rating_max": 5.0}
(ARTIFACT_DIR / "config.json").write_text(json.dumps(config, indent=2) + "\n")
training_history.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
print(f"Saved two-tower model and serving data to {ARTIFACT_DIR.resolve()}")

In [ ]:
# Optional reload after running the definition cells in a fresh kernel.
# checkpoint = torch.load(ARTIFACT_DIR / "two_tower.pt", map_location="cpu", weights_only=True)
# restored_model = TwoTowerModel(**checkpoint["model_config"])
# restored_model.load_state_dict(checkpoint["state_dict"])
# restored_model.to(device).eval()
# restored_features = np.load(ARTIFACT_DIR / "item_features.npy", allow_pickle=False)
# restored_vectors = np.load(ARTIFACT_DIR / "item_vectors.npy", allow_pickle=False)
# restored_catalog = pd.read_csv(ARTIFACT_DIR / "catalog.csv", dtype={"recipe_id": "string"})
# restored_history = pd.read_csv(ARTIFACT_DIR / "observed_ratings.csv", dtype={"user_id": "string"})
# restored_history["date"] = pd.to_datetime(restored_history["date"], utc=True)
# saved_config = json.loads((ARTIFACT_DIR / "config.json").read_text())
# restored_results = recommend_recipes(
#     restored_model, restored_catalog, restored_features, restored_history,
#     user_id=restored_history["user_id"].iloc[0], top_k=20, item_embeddings=restored_vectors,
#     max_history=saved_config["max_history"], half_life_days=saved_config["recency_half_life_days"])